# Create Digital Research Alliance of Canada Awards

Creates awards for the Digital Research Alliance of Canada (the Alliance) from the two funding
programmes for which the Alliance publishes recipient-level lists on alliancecan.ca:
- **DRI EDIA Champions pilot program** (call June 2024, projects Sept 2024 - March 31, 2025, up to
  CAD 35,000 each): 82 recipients, table on /en/initiatives/dri-investments
  (recipient, institution, project title).
- **Data Champions Pilot Project** (call January 2022, CAD 916,000 in total): 18 recipients on
  /en/opportunities/data-champions-pilot-project-call (applicant, team, project, institution).

**100 awards** (local run 2026-10-01). Both are capacity-building programmes for digital research
infrastructure (research data management, EDIA in DRI); kept under the 2026-10-01 scope rule.

**Not ingested:** Resource Allocation Competition (RAC) compute/storage allocations (in-kind; no
per-project list on the current site; held for a coordinator decision); institution-level DRI
investments (ARC renewal contributions etc.: news prose, no PI); the CoreTrustSeal certification
cohort (repository list, no PI); 2026 Research Software AI Enhancement recipients (prose, no PI).

**Prerequisites:**
- Run `scripts/local/alliancecan_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/alliancecan/alliancecan_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** the Alliance prints no award number for these programmes, and the
~84 citation stubs under F4320331257 are RAC allocation ids ("RRG 4073", "RPP 772",
"xvj-290-01"), so a stable synthetic key is shipped: `ALLIANCE-EDIA-2024-<applicant-slug>` /
`ALLIANCE-DATACHAMP-2022-<applicant-slug>`. 0 stubs collapse (expected).

**Amounts / dates:** per-recipient amounts are not published (amount NULL). `start_year` = award
year; `end_date` = the EDIA programme's published project end date (2025-03-31).

**lead_investigator:** the named recipient / principal applicant (split in Python, runbook §2.4.1),
institution as affiliation; one organisational applicant (Ocean Networks Canada) carries the
affiliation only.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320331257` (Alliance de recherche numérique du Canada / Digital Research Alliance of
  Canada, CA; ror 010r6td27; doi 10.13039/501100021202)

**Priority:** `548` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.alliancecan_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/alliancecan/alliancecan_projects.parquet`;

In [ ]:
%sql
SELECT programme, COUNT(*) as n FROM openalex.awards.alliancecan_raw GROUP BY programme;

## Step 1.5: Inspect raw data

In [ ]:
%sql
DESCRIBE openalex.awards.alliancecan_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.alliancecan_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320331257 is a Crossref-registered funder, so it MUST resolve to exactly 1 row in
`openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320331257;

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320331257 must have exactly one row in openalex.funders.funders') AS funder_ok
FROM openalex.funders.funders
WHERE funder_id = 4320331257;

## Step 2: Create Alliance Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.alliancecan_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320331257  -- Digital Research Alliance of Canada
),

lead AS (
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        funder_award_id,
        named_struct(
            'given_name', NULLIF(TRIM(lead_given_name), ''),
            'family_name', NULLIF(TRIM(lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) AS lead_investigator
    FROM openalex.awards.alliancecan_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    CAST(NULL AS STRING) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'grant' as funding_type,
    NULLIF(TRIM(g.programme), '') as funder_scheme,
    'alliancecan_funding_programmes' as provenance,
    CAST(NULL AS DATE) as start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
    TRY_CAST(g.year AS INT) as start_year,
    YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
    l.lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    array(l.lead_investigator) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.alliancecan_raw g
JOIN lead l ON l.funder_award_id = g.funder_award_id
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Shape check (§2.1.1): synthetic ALLIANCE-<programme>-<year>-<slug> keys only; ids unique.
SELECT
    assert_true(COUNT_IF(NOT funder_award_id RLIKE '^ALLIANCE-(EDIA|DATACHAMP)-20[0-9]{2}-[a-z0-9-]+$') = 0,
                'funder_award_id is not an ALLIANCE-<programme>-<year>-<slug> key') AS shape_ok,
    assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok
FROM openalex.awards.alliancecan_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'alliancecan_funding_programmes' AND priority = 548;

-- Insert into openalex_awards_raw with priority.
-- Priority 548: direct-from-funder ingest of the Alliance's published funding-programme recipients.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    548 as priority
FROM openalex.awards.alliancecan_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.alliancecan_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, start_year, end_date,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.alliancecan_awards LIMIT 20;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.alliancecan_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.alliancecan_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage (amounts are not published: pct_amount 0 by design).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(start_year) as has_start_year,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_affiliation
FROM openalex.awards.alliancecan_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'alliancecan_funding_programmes'
GROUP BY provenance, priority;